# 01 — Data Acquisition

**Scientific question:** Can compositional (MAGPIE) descriptors predict a material's density of
states at the Fermi level, DOS(E_F) — a cheap, DFT-derived pre-screening signal for conventional
electron-phonon superconductivity — and does that prediction generalize to unseen elemental
chemistries rather than just interpolating within familiar ones?

This notebook queries the Materials Project for experimentally observed metals, deduplicates by
composition, classifies each material's elemental family/period, takes a stratified sample, and
fetches DOS(E_F) for each sampled material. Output: `../data/metals_dos.csv`, consumed by
`02_eda_featurization.ipynb`.

**Requirements:** a free Materials Project API key in a `.env` file at the project root
(`MP_API_KEY=your_key_here`, see `../.env.example`), and internet access to
`api.materialsproject.org`. This notebook was not executed in the authoring environment (no
network access there) — run it locally with a valid key.


---
## Part 1 — Setup

In [ ]:
# Imports
import os
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from dotenv import load_dotenv

from mp_api.client import MPRester, MPRestError
from pymatgen.core import Element

print("Imports OK")


In [ ]:
# Materials Project API key, loaded from a .env file at the project root
# 1. pip install python-dotenv
# 2. Create a file named ".env" in the project root (one level up from notebooks/) containing:
#      MP_API_KEY=your_key_here
# 3. ".env" is already listed in .gitignore so the key never gets committed.

load_dotenv("../.env")  # reads .env at the project root

MP_API_KEY = os.environ.get("MP_API_KEY", "")
if not MP_API_KEY:
    raise ValueError(
        "MP_API_KEY not found. Make sure you have a .env file at the project root "
        "containing a line like: MP_API_KEY=your_key_here"
    )


---
## Part 2 — Query experimentally observed metals

Only metals have a physically meaningful DOS(E_F): for an insulator or semiconductor the Fermi level
sits in the gap, so DOS(E_F) is either exactly zero or an artifact of the DFT smearing scheme, not a
real physical quantity. We filter to `band_gap == 0` (metals) at the summary-search stage, before
doing the more expensive per-material DOS lookups.

We restrict to `theoretical=False` — experimentally observed, ICSD-matched materials — rather than
`is_stable=True`. This is a deliberate choice: it keeps the dataset to real, synthesized compounds
(the kind of material a follow-up electron-phonon coupling study would actually target), including
ones that are metastable relative to DFT's predicted hull, rather than every DFT-predicted candidate
regardless of whether it has ever been made.

In [ ]:
# Cache the raw query result to disk so re-running this notebook doesn't hit the
# Materials Project API again. Structure objects aren't plain-JSON/CSV-serializable, so this
# raw cache is kept as a pickle outside data/ (and is >50 MB, so it's gitignored). Delete the
# cache file to force a fresh query (e.g. if you change the filter criteria below).
QUERY_CACHE_PATH = "../cache/metals_query_cache.pkl"

if os.path.exists(QUERY_CACHE_PATH):
    df_all = pd.read_pickle(QUERY_CACHE_PATH)
    print(f"Loaded cached query results from {QUERY_CACHE_PATH}: {len(df_all)} materials "
          f"(delete this file to force a fresh query)")
else:
    FIELDS = [
        "material_id", "formula_pretty", "structure", "band_gap",
        "symmetry", "theoretical", "energy_above_hull",
    ]
    with MPRester(MP_API_KEY) as mpr:
        docs = mpr.materials.summary.search(
            band_gap=(0, 0), theoretical=False, fields=FIELDS,
        )
    df_all = pd.DataFrame([{
        "material_id": str(d.material_id),
        "formula": d.formula_pretty,
        "structure": d.structure,
        "crystal_system": d.symmetry.crystal_system.value if d.symmetry else None,
        "energy_above_hull": d.energy_above_hull,
    } for d in docs])
    os.makedirs(os.path.dirname(QUERY_CACHE_PATH), exist_ok=True)
    df_all.to_pickle(QUERY_CACHE_PATH)
    print(f"Queried {len(df_all)} experimentally observed metals, cached to {QUERY_CACHE_PATH}")


---
## Part 3 — Deduplicate to the lowest-energy polymorph per composition

MAGPIE descriptors are purely compositional — they don't see structure at all, so every polymorph of
the same formula (different space group, different calculation) produces an **identical** feature
vector. Leaving duplicates in would let a single composition contribute several (near-)identical rows
to Lasso feature selection and to cross-validation, silently inflating its influence.

We keep only the lowest-`energy_above_hull` entry per unique formula: the polymorph closest to (or on)
the hull is the one most likely to correspond to what was actually observed experimentally.

In [ ]:
before = len(df_all)

df_all = (
    df_all
    .sort_values("energy_above_hull", na_position="last")
    .drop_duplicates(subset="formula", keep="first")
    .reset_index(drop=True)
)

print(f"Deduplicated by composition: {before} -> {len(df_all)} unique formulas "
      f"(kept lowest energy_above_hull polymorph per formula)")


---
## Part 4 — Elemental family and period labels

Before deciding how many materials to actually pull full DOS objects for, we classify each one so we
can (a) sample in a way that keeps every group well represented and (b) reuse the same labels later
as `GroupKFold` groups. Each material's group is defined from its **dominant element** — the element
with the highest atomic fraction in the composition — using pymatgen's built-in element-category
flags for the family, and the element's periodic row for the period.

In [ ]:
def dominant_element(struct):
    """Element with the largest atomic fraction in the composition."""
    frac = struct.composition.get_el_amt_dict()
    return max(frac, key=frac.get)


def elemental_family(el_symbol):
    """Coarse elemental family used for GroupKFold grouping."""
    el = Element(el_symbol)
    # pymatgen renamed this property at some point (is_rare_earth_metal -> is_rare_earth);
    # check both so this works across pymatgen versions.
    is_rare_earth = getattr(el, "is_rare_earth", None)
    if is_rare_earth is None:
        is_rare_earth = getattr(el, "is_rare_earth_metal", el.is_lanthanoid or el.is_actinoid)
    if is_rare_earth:
        return "rare_earth"
    if el.is_alkaline:
        return "alkaline_earth"
    if el.is_alkali:
        return "alkali"
    if el.is_transition_metal:
        return "transition_metal"
    if el.is_post_transition_metal:
        return "post_transition_metal"
    if el.is_metalloid:
        return "metalloid"
    return "other"


df_all["dominant_element"] = df_all["structure"].apply(dominant_element)
df_all["family"] = df_all["dominant_element"].apply(elemental_family)
df_all["period"] = df_all["dominant_element"].apply(lambda e: Element(e).row)

print("Materials per elemental family:")
print(df_all["family"].value_counts())
print("\nMaterials per (family, crystal system):")
print(df_all.groupby(["family", "crystal_system"]).size())


---
## Part 5 — Stratified, capped sample

Rather than fetching DOS objects for every remaining metal in `df_all`, we take a stratified sample
capped at `CAP_PER_STRATUM` materials per (family × crystal system) combination. This bounds the
number of expensive lookups in Part 6 to a small, predictable multiple of the number of strata, and
ensures every group has comparable representation going into the `GroupKFold` evaluation later.

Raise `CAP_PER_STRATUM` if you want a larger dataset and can tolerate a longer Part 6 runtime; lower
it to iterate faster while developing the rest of the pipeline.

In [ ]:
CAP_PER_STRATUM = 300
RANDOM_STATE = 42

df = (
    df_all
    .groupby(["family", "crystal_system"], group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), CAP_PER_STRATUM), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

print(f"Sampled {len(df)} / {len(df_all)} metals "
      f"(capped at {CAP_PER_STRATUM} per family x crystal-system stratum)")
print(df.groupby("family").size())


---
## Part 6 — Fetch DOS objects and extract DOS(E_F)

For each sampled metal, we pull the computed `CompleteDos` object (GGA or GGA+U NSCF, as stored by
the Materials Project's electronic-structure endpoint) and linearly interpolate the **total** density
of states at the Fermi energy reported for that calculation.

We fetch in parallel with a thread pool (these are network-bound calls). Each worker thread gets its
own `MPRester` instance (created lazily on first use, one per thread rather than one per material) —
sharing a single client across threads is unsafe for this specific call, since it lazily registers a
cached Delta table on first use and concurrent threads can race to register it. The `try/finally`
ensures the executor shuts down cleanly (cancelling any still-pending lookups) even if the cell is
interrupted or a lookup raises.

In [ ]:
import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
from tqdm.auto import tqdm

# MPRester is not safe to share across threads for this particular call: get_dos_by_material_id
# registers a cached "total_dos" Delta table lazily on first use, and if two threads call it at
# the same moment on a *shared* client, both can see "not yet registered" and race to register it,
# raising DeltaError: "table already exists". Giving each worker THREAD its own client (created
# lazily, once per thread, via threading.local()) avoids the race without losing the parallelism -
# ThreadPoolExecutor reuses its 8 worker threads across all submitted tasks, so this creates only
# 8 MPRester instances total, not one per material.
_thread_local = threading.local()

def get_dos_at_fermi(mid):
    if not hasattr(_thread_local, "mpr"):
        _thread_local.mpr = MPRester(MP_API_KEY, mute_progress_bars=True)
    mpr = _thread_local.mpr
    # MPRestError here means no NSCF DOS calculation exists for this material -
    # a real, expected condition (not every metal in MP has electronic structure
    # data computed), not a bug. We skip it rather than let one missing entry
    # kill the whole batch. Any other exception type still propagates normally.
    try:
        dos = mpr.get_dos_by_material_id(mid)
    except MPRestError:
        return mid, None
    densities = dos.get_densities()          # spin channels already summed
    return mid, float(np.interp(dos.efermi, dos.energies, densities))


# Incremental cache, keyed by material_id, stored as plain JSON. Unlike caching the whole
# sampled dataframe, this survives changing CAP_PER_STRATUM or RANDOM_STATE later - only
# material_ids not already in the cache get fetched, and everything already looked up
# (across any previous run) is reused for free. Small (<50 MB) raw cache, so it lives in data/.
DOS_CACHE_PATH = "../data/dos_ef_cache.json"

if os.path.exists(DOS_CACHE_PATH):
    with open(DOS_CACHE_PATH) as f:
        dos_ef_map = json.load(f)
    print(f"Loaded {len(dos_ef_map)} cached DOS(E_F) values from {DOS_CACHE_PATH}")
else:
    dos_ef_map = {}

to_fetch = [mid for mid in df["material_id"] if mid not in dos_ef_map]
print(f"{len(df) - len(to_fetch)} already cached, fetching {len(to_fetch)} new materials...")

if to_fetch:
    executor = ThreadPoolExecutor(max_workers=8)
    try:
        futures = [executor.submit(get_dos_at_fermi, mid) for mid in to_fetch]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching DOS(E_F)"):
            mid, dos_ef = future.result()
            dos_ef_map[mid] = dos_ef
    finally:
        executor.shutdown(wait=True, cancel_futures=True)

    os.makedirs(os.path.dirname(DOS_CACHE_PATH), exist_ok=True)
    with open(DOS_CACHE_PATH, "w") as f:
        json.dump(dos_ef_map, f)
    print(f"Saved {len(dos_ef_map)} total cached DOS(E_F) values to {DOS_CACHE_PATH}")

df["dos_ef"] = df["material_id"].map(dos_ef_map)
n_ok = df["dos_ef"].notna().sum()
print(f"DOS(E_F) retrieved for {n_ok} / {len(df)} materials "
      f"({len(df) - n_ok} skipped: no NSCF DOS calculation available).")


---
## Save output

`structure` (a pymatgen `Structure` object) isn't CSV-serializable and isn't needed downstream —
composition-only MAGPIE featurization in `02_eda_featurization.ipynb` can be reconstructed from the
`formula` string alone via `pymatgen.core.Composition(formula)`. Everything else is plain scalar
columns, so we drop `structure` and write the rest to `../data/metals_dos.csv`.

In [ ]:
df_out = df.drop(columns=["structure"])
df_out = df_out[df_out["dos_ef"].notna()].reset_index(drop=True)

os.makedirs("../data", exist_ok=True)
df_out.to_csv("../data/metals_dos.csv", index=False)
print(f"Saved {len(df_out)} materials to ../data/metals_dos.csv")
df_out.head()
